<a href="https://colab.research.google.com/github/Cassio295/Cassio295/blob/master/noticias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Projeto de classificação de Sentimenos

In [ ]:
!pip install gensim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Pegando os arquivos dentro do drive

In [ ]:
!unzip "/content/drive/MyDrive/criticas-imdb.zip"

Archive:  /content/drive/MyDrive/criticas-imdb.zip
  inflating: criticas-imdb.csv       


In [ ]:
# Lendo os dados
import pandas as pd
criticas = pd.read_csv('criticas-imdb.csv')

In [ ]:
criticas.head()

,texto,sentimento
0,Eu fui e vi este filme ontem à noite depois de...,positivo
1,"O diretor do ator, Bill Paxton, segue sua prom...",positivo
2,Como um jogador de recreio com algum conhecime...,positivo
3,"Eu vi esse filme em uma prévia, e é delicioso....",positivo
4,Bill Paxton levou a verdadeira história do gol...,positivo


In [ ]:
criticas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49459 entries, 0 to 49458
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   texto       49459 non-null  object
 1   sentimento  49459 non-null  object
dtypes: object(2)
memory usage: 772.9+ KB


In [ ]:
criticas.isnull().sum()

,0
texto,0
sentimento,0


In [ ]:
import nltk as n
import string
n.download('punkt')
n.download('punkt_tab')

# Escrevendo função para gerar os tokens tratando a puntuação junto
def gerar_tokens(texto):
  texto = texto.lower()
  tokens = []

  for token in n.word_tokenize(texto):
    if token not in string.punctuation:
      tokens.append(token)
  return tokens


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Começando a tratar o texto

Como a proposta e usar o Word2Vec para a classificação não vou remover as stopwords dos tokens pois elas podem ajudar no contexto, para a utilização do Word2Vec manter as stopwords pode ser interessante.

In [ ]:
criticas.head()

,texto,sentimento
0,Eu fui e vi este filme ontem à noite depois de...,positivo
1,"O diretor do ator, Bill Paxton, segue sua prom...",positivo
2,Como um jogador de recreio com algum conhecime...,positivo
3,"Eu vi esse filme em uma prévia, e é delicioso....",positivo
4,Bill Paxton levou a verdadeira história do gol...,positivo


In [ ]:
!unzip '/content/cbow_s600.zip'
!unzip '/content/skip_s600.zip'

Archive:  /content/cbow_s600.zip
  inflating: cbow_s600.txt           
Archive:  /content/skip_s600.zip
  inflating: skip_s600.txt           


In [ ]:
# Importando os modelos pré-treinado do Word2vEC DO NILC
from gensim.models import KeyedVectors
model_skip = KeyedVectors.load_word2vec_format('/content/skip_s600.txt')
model_cbow = KeyedVectors.load_word2vec_format('/content/cbow_s600.txt')

In [ ]:
criticas.head(1)

,texto,sentimento
0,Eu fui e vi este filme ontem à noite depois de...,positivo


In [ ]:
# gerandos os tokens do meu texto
criticas['tokens'] = criticas['texto'].apply(lambda x: gerar_tokens(x))

In [ ]:
# função para pegar os vetores dos modelos
import numpy as np

def vetorizar(tokens, modelo):
    vetor = np.zeros(modelo.vector_size)
    for token in tokens:
        try:
            vetor += modelo[token]
        except KeyError:
            if token.isnumeric():
                fallback_num = "0" * len(token)
                if fallback_num in modelo:
                    vetor += modelo[fallback_num]
                elif "unknown" in modelo:
                    vetor += modelo["unknown"]
            elif "unknown" in modelo:
                vetor += modelo["unknown"]
    return vetor


In [ ]:
# Transformando a coluna sentimento em numerico com OnoHotEncoder
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()

criticas['senti'] = label.fit_transform(criticas['sentimento'])

In [ ]:
# Tirando os vetores do dataset
criticas['vetores_cbow'] = criticas['tokens'].apply(lambda x : vetorizar(x, model_cbow))
criticas['vetores_skip'] = criticas['tokens'].apply(lambda x : vetorizar(x, model_skip))

In [ ]:
# Separando meu conjunto de dados em treino e teste
from sklearn.model_selection import train_test_split
X_cbow = np.vstack(criticas['vetores_cbow'].values)
X_skip = np.vstack(criticas['vetores_skip'].values)
y = criticas.senti.values



# Dados de treino e teste para o c_bow
x_train_c, x_test_c, y_train, y_test = train_test_split(X_cbow, y, test_size=0.1, random_state=42)
x_train_s, x_test_s, y_train, y_test = train_test_split(X_skip, y, test_size=0.1,random_state=42)



In [ ]:
print(x_train_c.shape)
print(x_test_c.shape)
print(x_train_s.shape)
print(x_test_s.shape)
print(y_train.shape)
print(y_test.shape)


(44513, 600)
(4946, 600)
(44513, 600)
(4946, 600)
(44513,)
(4946,)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
classi = LogisticRegression(max_iter=250, random_state=42)
random = RandomForestClassifier(n_estimators=100, random_state=42)

random_c_bow = random.fit(x_train_c, y_train)
classi_c_bow = classi.fit(x_train_c, y_train)

In [ ]:
# Avaliando os modelos treinados com cbow_s600
y_pred_random_c_bow = random_c_bow.predict(x_test_c)
y_pred_classi_c_bow = classi_c_bow.predict(x_test_c)

print('-' * 20, 'Floresta Aleatoria', '-'*20)
print(classification_report(y_test, y_pred_random_c_bow))

print('-' * 20, 'Regressão Logistica', '-'*20)
print(classification_report(y_test, y_pred_classi_c_bow))





-------------------- Floresta Aleatoria --------------------
              precision    recall  f1-score   support

           0       0.77      0.73      0.75      2552
           1       0.73      0.76      0.74      2394

    accuracy                           0.75      4946
   macro avg       0.75      0.75      0.75      4946
weighted avg       0.75      0.75      0.75      4946

-------------------- Regressão Logistica --------------------
              precision    recall  f1-score   support

           0       0.84      0.82      0.83      2552
           1       0.81      0.83      0.82      2394

    accuracy                           0.83      4946
   macro avg       0.83      0.83      0.83      4946
weighted avg       0.83      0.83      0.83      4946



O modelo usando a regressão logistica performou melhor em comparação a floresta aleatoria. A seguir eu vou tentar outra abordagem aumentando os parametros de ambos os modelos e testando o skip_s600

In [ ]:
classi = LogisticRegression(max_iter=400)
random = RandomForestClassifier(n_estimators=200)

random_cbow = random.fit(x_train_c, y_train)
random_skip = random.fit(x_train_s, y_train)

classi_cbow = classi.fit(x_train_c, y_train)
classi_skip = classi.fit(x_train_s, y_train)

In [ ]:
y_pred_random_cbow
y_pred_random_skip

